# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [5]:
#!pip install pandas

In [3]:
import pandas as pd
df = pd.read_csv('data/Data.csv')

In [4]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [5]:
!pip install langchain_community

In [6]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [7]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=None)


In [17]:
prompt = ChatPromptTemplate.from_template(  "Can you describe the features and benefits of the product: {product}?"
 
)

In [18]:

chain = LLMChain(llm=llm, prompt=prompt)

In [19]:
product = "Kindle Paperwhite"
chain.run(product)

C:\Users\Admin\AppData\Local\Temp\ipykernel_78772\949943981.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chain.run(product)


'Sure! The Kindle Paperwhite is a popular e-reader device offered by Amazon. \n\nFeatures:\n1. High-resolution display with 300 ppi, so text appears crisp and sharp\n2. Built-in adjustable light allows for reading in all lighting conditions, even in direct sunlight\n3. Waterproof design, so you can read by the pool or in the bath without worry\n4. Long battery life, with weeks of use on a single charge\n5. Lightweight and portable design, perfect for reading on-the-go\n6. Built-in Audible compatibility, so you can switch between reading and listening to audiobooks seamlessly\n7. Customizable settings for font size, style, and layout preferences\n\nBenefits:\n1. Easy on the eyes: The high-resolution display and adjustable light make reading for extended periods comfortable and enjoyable.\n2. Versatile: The Kindle Paperwhite can hold thousands of books, making it a convenient way to carry and access a large library anywhere.\n3. Durable: The waterproof design and sturdy construction ensu

## SimpleSequentialChain

In [20]:
from langchain.chains import SimpleSequentialChain

In [22]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
   "Can you describe the features and benefits of the product: {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [23]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
   "Based on the following product description, write a short and engaging customer review:\n\n{product_description}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [24]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [25]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
The Kindle Paperwhite is a popular e-reader device produced by Amazon. Some of its key features and benefits include:

1. High-resolution display: The Kindle Paperwhite features a 300 ppi glare-free display that mimics the look of real paper, making reading for extended periods more comfortable.

2. Adjustable built-in light: The device has a built-in front light that can be adjusted for reading in any lighting conditions, from bright sunlight to complete darkness.

3. Waterproof design: The Kindle Paperwhite is IPX8 rated, which means it can withstand being submerged in water up to 2 meters deep for up to 60 minutes. This makes it a great option for reading by the pool or at the beach.

4. Long battery life: The Kindle Paperwhite can last for weeks on a single charge, allowing you to read for extended periods without needing to constantly recharge.

5. Huge selection of books: With access to the Amazon Kindle store, users have a vast lib

"I absolutely love my Kindle Paperwhite! The high-resolution display makes reading for hours on end a breeze, and the adjustable built-in light is perfect for reading in any setting. The fact that it's waterproof is a game-changer for me, as I can now read by the pool without worry. The long battery life is also a huge plus, as I can go weeks without needing to recharge. With a huge selection of books available in the Kindle store, I always have something new to read. Plus, the lightweight and portable design make it easy to take with me wherever I go. Overall, the Kindle Paperwhite has exceeded my expectations and I would highly recommend it to anyone looking for a top-notch e-reader experience."

**Repeat the above twice for different products**

## SequentialChain

In [26]:
from langchain.chains import SequentialChain

In [37]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
  "Translate the following product review into {target_language}:\n\n{review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="translated_review" #Give a name to your output
                    )


In [38]:
second_prompt = ChatPromptTemplate.from_template(
   "Summarize the following product review in 1-2 sentences:\n\n{review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary" #give a name to this output
                    )


In [39]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "Translate the following product review to {target_language}:\n\n{review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="translated_review_final"
                      )


In [40]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        "Based on the translated review:\n\n{translated_review_final}\n\n"
    "and this summary:\n\n{summary}\n\n"
    "Write a friendly follow-up message asking the user if they found this helpful and suggest checking out more similar products."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [41]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review", "target_language"],
    output_variables=["translated_review", "summary", "translated_review_final", "followup_message"],
    verbose=True
)

In [43]:
review = df.Review[5]
output = overall_chain({
    "review": review,
    "target_language": "English"  # or "French", "Arabic", etc.
})



> Entering new SequentialChain chain...

> Finished chain.


**Repeat the above twice for different products or reviews**

## Router Chain

In [44]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [45]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [46]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [47]:
llm = ChatOpenAI(temperature=0)

In [48]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [49]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [50]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [51]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [52]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\Admin\AppData\Local\Temp\ipykernel_78772\3038952769.py:1: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(router_chain=router_chain,


In [53]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This radiation is characterized by a continuous spectrum of wavelengths and intensities, with the peak intensity shifting to shorter wavelengths as the temperature of the black body increases. Black body radiation plays a key role in understanding concepts such as thermal radiation and the quantization of energy in quantum mechanics."

In [54]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [55]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells. \n\nAdditionally, DNA is constantly being used by cells to carry out processes such as protein synthesis, cell division, and repair. Having DNA in every cell allows for the coordination of these processes and ensures that the organism functions properly as a whole. \n\nIn summary, every cell in our body contains DNA because it is essential for the proper functioning and development of all living organisms.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [56]:
chain.run("how to an make iced latte ? ")



> Entering new MultiPromptChain chain...
None: {'input': 'how to an make iced latte ?'}
> Finished chain.


"To make an iced latte, you will need the following ingredients:\n\n- Espresso or strong brewed coffee\n- Milk (dairy or non-dairy)\n- Ice cubes\n- Sweetener (optional)\n\nHere's how to make an iced latte:\n\n1. Brew a shot of espresso or strong coffee. You can use a coffee machine, espresso machine, or a French press to make the coffee.\n\n2. Fill a glass with ice cubes.\n\n3. Pour the espresso or coffee over the ice cubes in the glass.\n\n4. In a separate container, froth or steam the milk. You can use a milk frother, a handheld frother, or a French press to froth the milk.\n\n5. Pour the frothed milk over the coffee and ice in the glass.\n\n6. Stir the coffee and milk together to combine.\n\n7. Add sweetener to taste, if desired.\n\n8. Enjoy your refreshing iced latte!"

In [57]:
chain.run("how to make a burger?")



> Entering new MultiPromptChain chain...
None: {'input': 'how to make a burger?'}
> Finished chain.


'To make a classic burger, you will need the following ingredients:\n\n- Ground beef\n- Salt and pepper\n- Hamburger buns\n- Cheese slices\n- Lettuce\n- Tomato slices\n- Onion slices\n- Pickles\n- Ketchup\n- Mustard\n\nInstructions:\n\n1. Start by shaping the ground beef into patties. Season each patty with salt and pepper on both sides.\n\n2. Preheat a grill or skillet over medium-high heat. Cook the patties for about 4-5 minutes on each side, or until they reach your desired level of doneness.\n\n3. While the patties are cooking, toast the hamburger buns on the grill or in a toaster.\n\n4. Once the patties are cooked, place a slice of cheese on top of each patty and allow it to melt slightly.\n\n5. Assemble your burger by placing a patty with cheese on the bottom half of a toasted bun. Top with lettuce, tomato, onion, pickles, ketchup, and mustard.\n\n6. Place the top half of the bun on top of the toppings and serve hot.\n\nEnjoy your delicious homemade burger!'